In [1]:

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import BertTokenizer, BertModel
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import numpy as np
import json
from scipy.stats import pearsonr


In [2]:
def read_jsonl(path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            data.append(json.loads(line))
    return data


def extract_aspect_level_samples(raw_data):
    samples = []
    for item in raw_data:
        sample_id = item["ID"]
        text = item["Text"]

        # Search for all possible annotation keys
        annotations = item.get("Quadruplet", item.get("Triplet", item.get("Aspect_VA", [])))
        for ann in annotations:
            v, a = ann["VA"].split("#")
            samples.append({
                "id": sample_id,
                "text": text,
                "aspect": ann["Aspect"],
                "valence": float(v),
                "arousal": float(a)
            })

        if "Aspect" in item and not annotations:
            for aspect in item["Aspect"]:
                samples.append({
                    "id": sample_id,
                    "text": text,
                    "aspect": aspect,
                    "valence": None,
                    "arousal": None
                })
    return samples

train_samples = extract_aspect_level_samples(read_jsonl("eng_laptop_train_alltasks.jsonl"))
dev_samples = extract_aspect_level_samples(read_jsonl("eng_laptop_dev_task1.jsonl"))
test_samples = extract_aspect_level_samples(read_jsonl("eng_laptop_test_task1.jsonl"))


print(f"Train: {len(train_samples)}, Dev: {len(dev_samples)}, Test: {len(test_samples)}")

Train: 5773, Dev: 275, Test: 1421


In [3]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def build_model_input(text, aspect):
    if aspect == "NULL":
        aspect = "overall"
    return text.strip() + " [SEP] " + aspect.strip()

class CDSA_Dataset(Dataset):
    def __init__(self, samples, tokenizer, max_length=128, is_test=False):
        self.samples = samples
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.is_test = is_test

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        model_input = build_model_input(sample["text"], sample["aspect"])

        encoding = self.tokenizer(
            model_input, padding="max_length", truncation=True,
            max_length=self.max_length, return_tensors="pt"
        )

        item = {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0)
        }

        if not self.is_test:
            item["labels"] = torch.tensor([sample["valence"], sample["arousal"]], dtype=torch.float)

            # --- NEW: Contrastive Label Generation ---
            # Assuming Valence is 1-9. < 4.5 is Negative (0), > 5.5 is Positive (2), else Neutral (1)
            v = sample["valence"]
            if v < 4.5:
                c_class = 0
            elif v > 5.5:
                c_class = 2
            else:
                c_class = 1
            item["contrastive_label"] = torch.tensor(c_class, dtype=torch.long)

        return item

train_loader = DataLoader(CDSA_Dataset(train_samples, tokenizer), batch_size=16, shuffle=True)
dev_loader = DataLoader(CDSA_Dataset(dev_samples, tokenizer), batch_size=16, shuffle=False)
test_loader = DataLoader(CDSA_Dataset(test_samples, tokenizer, is_test=True), batch_size=16, shuffle=False)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [4]:

class SupConLoss(nn.Module):
    def __init__(self, temperature=0.07):
        super(SupConLoss, self).__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        # features shape: (batch_size, hidden_dim)
        # normalize features for cosine similarity
        features = F.normalize(features, p=2, dim=1)

        batch_size = features.shape[0]
        labels = labels.contiguous().view(-1, 1)

        # Create a mask that is 1 if two samples have the same label, 0 otherwise
        mask = torch.eq(labels, labels.T).float().to(features.device)

        # Calculate dot product (cosine similarity since vectors are normalized)
        anchor_dot_contrast = torch.div(torch.matmul(features, features.T), self.temperature)

        # For numerical stability
        logits_max, _ = torch.max(anchor_dot_contrast, dim=1, keepdim=True)
        logits = anchor_dot_contrast - logits_max.detach()

        # Remove self-comparisons (diagonal)
        logits_mask = torch.scatter(
            torch.ones_like(mask), 1,
            torch.arange(batch_size).view(-1, 1).to(features.device), 0
        )
        mask = mask * logits_mask

        # Compute log probabilities
        exp_logits = torch.exp(logits) * logits_mask
        log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True) + 1e-9)

        # Compute mean of log-likelihood over positive examples
        mask_pos_pairs = mask.sum(1)
        mask_pos_pairs = torch.where(mask_pos_pairs < 1e-6, 1, mask_pos_pairs)
        mean_log_prob_pos = (mask * log_prob).sum(1) / mask_pos_pairs

        # Final loss
        loss = -mean_log_prob_pos
        return loss.mean()

In [5]:
# ==========================================
# CELL 4: MODEL ARCHITECTURE
# ==========================================
class CDSA_HybridModel(nn.Module):
    def __init__(self):
        super(CDSA_HybridModel, self).__init__()
        self.bert = BertModel.from_pretrained("bert-base-uncased")
        hidden_size = self.bert.config.hidden_size

        # 1. Projection Head for Contrastive Learning
        # (Maps 768-dim down to 128-dim for stable contrastive math)
        self.projection_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 128)
        )

        # 2. Regression Head for Exact VA Prediction
        self.regressor = nn.Linear(hidden_size, 2)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)

        # We use the [CLS] token. Because of the [SEP] aspect token,
        # BERT inherently contextualizes this [CLS] token around the aspect.
        context_vector = outputs.last_hidden_state[:, 0, :]

        # Branch A: Get exact numerical predictions
        va_predictions = self.regressor(context_vector)

        # Branch B: Get contrastive embeddings
        contrastive_features = self.projection_head(context_vector)

        return va_predictions, contrastive_features

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CDSA_HybridModel().to(device)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
import torch
if torch.cuda.is_available():
    print(f"Success! You are running on a GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU detected. Go back to Runtime -> Change runtime type.")

Success! You are running on a GPU: Tesla T4


In [7]:

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
mse_criterion = nn.MSELoss()
supcon_criterion = SupConLoss(temperature=0.1)

alpha = 0.3 # 30% contrastive focus, 70% exact regression focus

def train_epoch(model, dataloader):
    model.train()
    total_loss, total_mse, total_scl = 0, 0, 0

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        c_labels = batch["contrastive_label"].to(device)

        optimizer.zero_grad()

        # Forward pass
        preds, contrastive_feats = model(input_ids, attention_mask)

        # Calculate individual losses
        loss_mse = mse_criterion(preds, labels)
        loss_scl = supcon_criterion(contrastive_feats, c_labels)

        # Combine Loss
        loss = ((1 - alpha) * loss_mse) + (alpha * loss_scl)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_mse += loss_mse.item()
        total_scl += loss_scl.item()

    return total_loss / len(dataloader), total_mse / len(dataloader)

def eval_epoch(model, dataloader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in dataloader:
            preds, _ = model(batch["input_ids"].to(device), batch["attention_mask"].to(device))
            all_preds.append(preds.cpu().numpy())
            all_labels.append(batch["labels"].numpy())

    all_preds = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)

    v_corr = pearsonr(all_preds[:, 0], all_labels[:, 0])[0]
    a_corr = pearsonr(all_preds[:, 1], all_labels[:, 1])[0]
    return v_corr, a_corr

epochs = 7
best_v_corr = 0

for epoch in range(epochs):
    avg_loss, avg_mse = train_epoch(model, train_loader)
    v_corr, a_corr = eval_epoch(model, dev_loader)

    if v_corr > best_v_corr:
        best_v_corr = v_corr
        torch.save(model.state_dict(), "best_hybrid_model.pt")
        print(">>> Model Saved")

    print(f"Epoch {epoch+1} | Total Loss: {avg_loss:.4f} | MSE Loss: {avg_mse:.4f}")
    print(f"Valence Corr: {v_corr:.4f} | Arousal Corr: {a_corr:.4f}\n" + "-"*40)

>>> Model Saved
Epoch 1 | Total Loss: 2.0919 | MSE Loss: 1.9398
Valence Corr: 0.8365 | Arousal Corr: 0.4063
----------------------------------------
>>> Model Saved
Epoch 2 | Total Loss: 1.1060 | MSE Loss: 0.5984
Valence Corr: 0.8605 | Arousal Corr: 0.4561
----------------------------------------
>>> Model Saved
Epoch 3 | Total Loss: 0.9706 | MSE Loss: 0.4465
Valence Corr: 0.8783 | Arousal Corr: 0.4887
----------------------------------------
>>> Model Saved
Epoch 4 | Total Loss: 0.9059 | MSE Loss: 0.3682
Valence Corr: 0.8862 | Arousal Corr: 0.5049
----------------------------------------
>>> Model Saved
Epoch 5 | Total Loss: 0.8500 | MSE Loss: 0.3057
Valence Corr: 0.8992 | Arousal Corr: 0.5235
----------------------------------------
Epoch 6 | Total Loss: 0.8089 | MSE Loss: 0.2655
Valence Corr: 0.8887 | Arousal Corr: 0.5072
----------------------------------------
Epoch 7 | Total Loss: 0.7784 | MSE Loss: 0.2405
Valence Corr: 0.8838 | Arousal Corr: 0.5101
------------------------------